# LLM Querying Pipeline

In [ ]:
import os
import time
import pandas as pd

GEMINI_API_KEY = "I AM NOT DUMB"
OLLAMA_API_KEY = "BUT I LOVE HARD CODING MY API KEYS IN NOTEBOOKS"
ANTHROPIC_API_KEY = "WE BALL" # add when ready
OPENAI_API_KEY    = "I AM NOT DUMB EITHER"

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = OLLAMA_API_KEY

INPUT_CSV  = r"C:\Users\tanib\Downloads\DATA\Finally\FINALLYV2\paraphrased_prompts.csv"
OUTPUT_CSV = r"C:\Users\tanib\Downloads\DATA\Finally\FINALLYV2\llm_responses.csv"

MODELS = [
    ("claude-sonnet-4-6", "anthropic"),
    ("gpt-4o",                   "openai"),
    ("gemini-3.1-pro-preview",   "gemini"),
    ("gemma4:31b-cloud",         "ollama"),
    ("gpt-oss:120b-cloud",       "ollama"),
]

print(f"Configured {len(MODELS)} target models")

In [ ]:
from ollama import Client as OllamaClient
import google.genai as genai
import anthropic
from openai import OpenAI

gemini_client = genai.Client(api_key=GEMINI_API_KEY)

ollama_client = OllamaClient(
    host="https://ollama.com",
    headers={"Authorization": f"Bearer {OLLAMA_API_KEY}"},
)

anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

openai_client = OpenAI(api_key=OPENAI_API_KEY)

print("Clients ready")

In [ ]:
def query_gemini(model_name, prompt):
    response = gemini_client.models.generate_content(
        model=model_name,
        contents=prompt,
    )
    return response.text


def query_ollama(model_name, prompt):
    response = ollama_client.chat(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.message.content


def query_anthropic(model_name, prompt):
    resp = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers={
            "x-api-key": ANTHROPIC_API_KEY,
            "anthropic-version": "2023-06-01",
            "content-type": "application/json",
        },
        json={
            "model": model_name,
            "max_tokens": 4096,
            "messages": [{"role": "user", "content": prompt}],
        },
        timeout=120,
    )
    resp.raise_for_status()
    return resp.json()["content"][0]["text"]


def query_openai(model_name, prompt):
    response = openai_client.responses.create(
        model=model_name,
        input=prompt,
    )
    return response.output_text


def query_model(model_name, provider, prompt):
    if provider == "gemini":
        return query_gemini(model_name, prompt)
    if provider == "ollama":
        return query_ollama(model_name, prompt)
    if provider == "anthropic":
        return query_anthropic(model_name, prompt)
    if provider == "openai":
        return query_openai(model_name, prompt)
    raise ValueError(f"Unknown provider: {provider}")


print("Query functions defined")

In [ ]:
# Smoke test - one prompt per active model. Run this once before the full loop.
TEST_PROMPT = "Reply with exactly one word: PONG"

for model_name, provider in MODELS:
    try:
        out = query_model(model_name, provider, TEST_PROMPT)
        print(f"OK  | {model_name:30s} -> {out[:80]!r}")
    except Exception as e:
        print(f"ERR | {model_name:30s} -> {type(e).__name__}: {e}")

In [ ]:
def load_done(output_csv):
    if not os.path.exists(output_csv):
        return set()
    df = pd.read_csv(output_csv)
    return set(zip(df["row_index"], df["target_model"]))


def append_row(output_csv, row_dict):
    df = pd.DataFrame([row_dict])
    header = not os.path.exists(output_csv)
    df.to_csv(output_csv, mode="a", header=header, index=False, encoding="utf-8")


print("Checkpoint helpers ready")

In [ ]:
seeds = pd.read_csv(INPUT_CSV)
done = load_done(OUTPUT_CSV)

total = len(seeds) * len(MODELS)
completed = len(done)
print(f"Total: {total} | Already done: {completed} | Remaining: {total - completed}")

for _, row in seeds.iterrows():
    for model_name, provider in MODELS:
        key = (row["row_index"], model_name)
        if key in done:
            continue

        try:
            response_text = query_model(model_name, provider, row["paraphrased_prompt"])
            error = ""
        except Exception as e:
            response_text = ""
            error = f"{type(e).__name__}: {e}"

        append_row(OUTPUT_CSV, {
            "row_index":          row["row_index"],
            "mitre_id":           row["mitre_id"],
            "mitre_name":         row["mitre_name"],
            "technique":          row["technique"],
            "paraphrased_prompt": row["paraphrased_prompt"],
            "target_model":       model_name,
            "model_response":     response_text,
            "error":              error,
            "timestamp":          time.strftime("%Y-%m-%d %H:%M:%S"),
        })

        completed += 1
        status = "OK" if not error else "ERR"
        print(f"[{completed}/{total}] {status} | {row['mitre_id']} | {row['technique']} | {model_name}")

        if error:
            time.sleep(2)

print("\nDone.")

In [ ]:
# Quick summary of what was collected
df = pd.read_csv(OUTPUT_CSV)
print(f"Total rows: {len(df)}")
print(f"\nRows per model:\n{df['target_model'].value_counts()}")
print(f"\nError rate per model:\n{(df['error'] != '').groupby(df['target_model']).mean()}")